# ⚙️ Configuration & Export

How to configure parsing, mask proper nouns, and build multi-project datasets.

## 🎯 Goals of this notebook

1. **Understand RunConfig options** — learn every toggle the parser exposes
2. **Compare outputs** — see how dropping damaged signs or masking POS changes transliterations
3. **Build a multi-project dataset** — parse several corpora and combine into one DataFrame
4. **Export** — save combined data as JSONL and CSV for downstream use
5. **Quick analysis** — explore the combined dataset by project, provenance, and period

## Configuration Options

Before we start, if the default data paths aren't correct for your machine, you can re-configure them:
By default, `DATA_DIR` is set to `<repo_root>/enriched_data`. If you have it somewhere else (like `D:/thesis_data`):

In [1]:
import oracc_parser.settings as settings
from pathlib import Path

# settings.DATA_DIR = Path("D:/my_custom_oracc_data")

## 1. RunConfig options

| Option | Default | What it does |
|---|---|---|
| `limit` | `None` | Only parse first N texts (good for testing) |
| `max_break_fraction` | `1.0` | **Word-level**: fraction of broken signs tolerated before a word is replaced with `X` in transliteration / normalization / lemmatization |
| `drop_missing` | `False` | **Sign-level**: remove signs marked `[x]` (completely broken) from **Unicode cuneiform output only** |
| `drop_damaged` | `False` | **Sign-level**: remove signs marked `⸢x⸣` (partially damaged) from **Unicode cuneiform output only** |
| `mask_pos` | `[]` | Replace words of certain POS with the tag name |
| `languages` | `["Akkadian"]` | Which languages to process |
| `USE_CACHE` | `True` | Use cached results if available |

### Discovering Valid Configurations
If you want to know what values are valid for `mask_pos`, `languages`, or `periods`, you can query the bundled reference data directly:

In [1]:
from oracc_parser.pipeline import reference_data

# See valid POS tags for masking
pos_df = reference_data.get_pos_tags()
print('Valid POS tags (first 15):')
print(pos_df['POS-tag'].head(15).tolist())
print()

# See valid languages
lang_df = reference_data.get_languages()
print('Valid Languages:')
print(lang_df['lang'].tolist())


Valid POS tags (first 15):
['MN', 'n', 'u', 'N', 'CN', 'AJ', 'PRP', 'V', 'IP', 'PN', 'MOD', 'RN', 'GN', 'REL', 'AV']

Valid Languages:
['sux', 'akk', 'akk-x-neoass', 'akk-x-stdbab', 'akk-x-neobab', 'akk-x-midass', 'akk-x-mbperi', 'qpc', 'akk-x-ltebab', 'akk-x-oldbab', 'akk-949', 'uga-040', 'sux-x-emesal', 'xur', 'peo', 'xur-946', 'hit', 'akk-x-midbab', 'arc', 'arc-949', 'elx', 'akk-x-stdbab-949', 'akk-x-earakk', 'akk-x-oldass', 'uga', 'xhu', 'hit-946', 'akk-936', 'xur-944', 'qca', 'qeb', 'akk-x-mbperi-949', 'akk-x-oldakk', 'qcu', 'qpe', 'grc', 'qur', 'akk-935', 'xhu-040', 'akk-x-neoass-949', 'xhu-946', 'sux-947', 'egy-020', 'sux-x-gloss', 'hlu', 'akk-x-neobab-949', 'sux-x-syll', 'hit-040', 'akk-x-oldbab-949']


## 2. Word-level vs sign-level break filtering

`RunConfig` exposes **two independent levels** of break filtering that operate on different granularities and affect different outputs.

### Word-level — `max_break_fraction` (0.0 – 1.0)

- Each word has a `break_perc`: the **fraction of its signs** that are broken or missing (averaged across all signs in the word).
- Words whose `break_perc` **exceeds** `max_break_fraction` are replaced wholesale with `X`.
- Affects → **transliteration**, **normalization**, **lemmatization**.
- `1.0` (default) → keep all words regardless of damage.
- `0.0` → replace any word that has even one broken sign.

### Sign-level — `drop_missing` / `drop_damaged`

- Operates **sign-by-sign**, not word-by-word.
- `drop_missing=True` removes signs marked `[x]` (completely lost).
- `drop_damaged=True` removes signs marked `⸢x⸣` (partially legible).
- Affects → **Unicode cuneiform output only**.

> ⚠️ **The two levels are independent and produce results that may not align.**
> A word kept intact in the transliteration (because its *average* damage is
> below `max_break_fraction`) may still have individual signs stripped from
> the Unicode cuneiform output by `drop_missing` / `drop_damaged`, and
> vice-versa. Do not expect the text outputs and the Unicode cuneiform to
> be aligned token-for-token when using these parameters.

In [1]:
from oracc_parser import parse_project, RunConfig, get_transliterations, get_unicode_texts

PROJECT = "saao/saa01"

# Strict word-level filtering: words with >30% broken signs → replaced with X
rec_strict = parse_project(PROJECT, config=RunConfig(limit=2, max_break_fraction=0.3))

# Liberal (default): keep all words regardless of damage
rec_liberal = parse_project(PROJECT, config=RunConfig(limit=2, max_break_fraction=1.0))

print("=== Transliteration with max_break_fraction=0.3 (strict) ===")
for _, r in get_transliterations(rec_strict).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Transliteration with max_break_fraction=1.0 (liberal, default) ===")
for _, r in get_transliterations(rec_liberal).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Unicode cuneiform is unaffected by max_break_fraction ===")
print("(drop_missing / drop_damaged control sign-level filtering here)")
for _, r in get_unicode_texts(rec_strict).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

11:18:49 | INFO    | oracc_parser | Already downloaded: G:\My Drive\GitHub\oracc-parser\enriched_data\jsonzip\saao-saa01.zip
Parsing saao/saa01: 100%|██████████| 2/2 [00:00<00:00, 11.20it/s]
11:18:50 | INFO    | oracc_parser | Loaded 2/2 tablets from cache, parsed 0 new
11:18:50 | INFO    | oracc_parser | Already downloaded: G:\My Drive\GitHub\oracc-parser\enriched_data\jsonzip\saao-saa01.zip
Parsing saao/saa01: 100%|██████████| 2/2 [00:00<00:00, 11.00it/s]
11:18:52 | INFO    | oracc_parser | Loaded 2/2 tablets from cache, parsed 0 new


=== Transliteration with max_break_fraction=0.3 (strict) ===
  saao/saa01_P314227:
    X X X
X X X DI-mu a-⸢na⸣
X X X X ⸢be⸣-li₂ X
X X X X X ina IGI
X X X X X X a-na-ku
X X X X up]-⸢ta⸣-at-ti\t-iu-u
X X X X 
    tokens_without_broken: 18/107
  saao/saa01_P313458:
    X X EN-⸢ia⸣ ARAD-⸢ka⸣ X
X ⸢šul⸣-mu a-na X EN⸣-ia₂ {d}AG\d u X X X X
X UD-MEŠ SUD-MEŠ ⸢ṭu⸣-ub ŠA₃-bi ṭu-ub X X X
X dan-nu
    tokens_without_broken: 107/179

=== Transliteration with max_break_fraction=1.0 (liberal, default) ===
  saao/saa01_P314227:
    [a-na LUGAL be-li₂]-ia₂
[ARAD-ka {1}ki-ṣir—aš-šur lu] DI-mu a-⸢na⸣
[LUGAL be-li₂-ia₂ ša LUGAL] ⸢be⸣-li₂ iš-⸢pur-an-ni⸣
[
    tokens_without_broken: 62/107
  saao/saa01_P313458:
    [a-na LUGAL] EN-⸢ia⸣ ARAD-⸢ka⸣ {1}hu-un-ni-i]
[lu-u] ⸢šul⸣-mu a-na ⸢LUGAL EN⸣-ia₂ {d}AG\d u {⸢d⸣}[AMAR.UTU a-na LUGAL EN
    tokens_without_broken: 167/179

=== Unicode cuneiform is unaffected by max_break_fraction ===
(drop_missing / drop_damaged control sign-level filtering here)
  saao/saa01_P

## 3. POS masking reference

| Tag | Meaning |
|---|---|
| `PN` | Personal Name |
| `DN` | Divine Name |
| `GN` | Geographical Name |
| `RN` | Royal Name |
| `TN` | Temple Name |

## 4. Build a multi-project dataset

In [4]:
import pandas as pd
from pathlib import Path
from oracc_parser import get_full_flat_table, get_metadata_table
from oracc_parser.settings import DATA_DIR, OUTPUT_DIR

# Find all downloaded projects
zip_dir = DATA_DIR / "jsonzip"
if not zip_dir.exists():
    zip_dir = Path("../data/jsonzip")

downloaded = sorted([f.stem for f in zip_dir.glob("*.zip")]) if zip_dir.exists() else []
print(f"Found {len(downloaded)} downloaded projects")
print(f"First 10: {downloaded[:10]}")

Found 149 downloaded projects
First 10: ['adsd', 'adsd-adart1', 'adsd-adart2', 'adsd-adart3', 'adsd-adart5', 'adsd-adart6', 'aemw', 'aemw-alalakh-idrimi', 'aemw-amarna', 'aemw-ugarit']


In [5]:
# Parse a few projects and combine
PROJECTS = ["saao/saa01", "saao/saa05"]  # change these to what you want
config = RunConfig(limit=3, max_break_fraction=.5, drop_missing=True)  # small limit for demo — remove for full parse

all_dfs = []
for project in PROJECTS:
    print(f"Parsing {project}...")
    try:
        records = parse_project(project, config=config)
        df = get_full_flat_table(records)
        all_dfs.append(df)
        print(f"  → {len(records)} tablets")
    except Exception as e:
        print(f"  → Error: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"\n✓ Combined: {len(combined)} tablets from {len(PROJECTS)} projects")
display(combined)

13:21:04 | INFO    | oracc_parser | Already downloaded: G:\My Drive\GitHub\oracc-parser\enriched_data\jsonzip\saao-saa01.zip


Parsing cams/gkab...


Parsing saao/saa01: 100%|██████████| 3/3 [00:00<00:00, 20.05it/s]
13:21:05 | INFO    | oracc_parser | Loaded 3/3 tablets from cache, parsed 0 new
13:21:05 | INFO    | oracc_parser | Already downloaded: G:\My Drive\GitHub\oracc-parser\enriched_data\jsonzip\saao-saa05.zip


  → 3 tablets
Parsing saao/saa05...


Parsing saao/saa05: 100%|██████████| 3/3 [00:00<00:00, 17.24it/s]
13:21:06 | INFO    | oracc_parser | Loaded 3/3 tablets from cache, parsed 0 new


  → 3 tablets

✓ Combined: 6 tablets from 2 projects


id     project  text_id                  genre provenance  \
0  saao/saa01_P314227  saao/saa01  P314227  Administrative Letter    Nineveh   
1  saao/saa01_P313458  saao/saa01  P313458  Administrative Letter    Nineveh   
2  saao/saa01_P334317  saao/saa01  P334317  Administrative Letter    Nineveh   
3  saao/saa05_P334831  saao/saa05  P334831  Administrative Letter    Nineveh   
4  saao/saa05_P314281  saao/saa05  P314281  Administrative Letter    Nineveh   
5  saao/saa05_P334186  saao/saa05  P334186  Administrative Letter    Nineveh   

         period  start_year  end_year  \
0  Neo-Assyrian        -721      -705   
1  Neo-Assyrian        -721      -705   
2  Neo-Assyrian        -721      -705   
3  Neo-Assyrian        -721      -705   
4  Neo-Assyrian        -717      -706   
5  Neo-Assyrian        -721      -705   

                                     transliteration  \
0  X X X\nX X X DI-mu a-⸢na⸣\nX X X X ⸢be⸣-li₂ iš...   
1  X X EN-⸢ia⸣ ARAD-⸢ka⸣ X\nX ⸢šul⸣-mu a-na ⸢LUGA...   
2  13 KUŠ₃ mu-lu-u 04 KUŠ₃ DAGAL E₂—dan-ni\nX X m...   
3  X DI]-mu a-[na X X\nX ⸢UGU⸣ UDU-MEŠ ⸢ša⸣ X X X...   
4  ⸢nu-uk\td 01 me 20 {giš}UR₃⸣-MEŠ\m\nša\ydm LUG...   
5  a-na LUGAL\p be-li₂-ia\nARAD-ka {1}aš-šur—BAD₃...   

                                       normalization  \
0  X X X\nX X X šulmu ana\nX X X X bēlī išpuranni...   
1  X X bēlīya urdaka X\nX šulmu ana šarri bēlīya ...   
2  UNK ammāti mūlû UNK ammāti rupšu bēt danni\nX ...   
3  X šulmu ana X X\nX muhhi immerē ša X X X X\nX ...   
4  nuk UNK mē UNK gušūrī\nša šarri alê\nšû issuhu...   
5  ana šarri bēlīya\nurdaka Aššur-dur-paniya\nlū ...   

                                       lemmatization  \
0  X X X\nX X X šulmu ana\nX X X X bēlu šapāru\nX...   
1  X X bēlu ardu X\nX šulmu ana šarru bēlu Nabu u...   
2  UNK ammatu mūlû UNK ammatu rupšu bītu dannu\nX...   
3  X šulmu ana X X\nX muhhu immeru ša X X X X\nX ...   
4  nuk UNK meʾatu UNK gušūru\nša šarru ali\nšū sa...   
5  ana šarru bēlu\nardu Aššur-dur-paniya\nlū šulm...   

                                             unicode  \
0  𒐊\n𒁲𒈬 𒀀𒈾\n𒁁𒉌 𒅖𒁓𒀭𒉌\n𒅀 𒀸 𒅆\n𒀀𒈾𒆪\n𒋫𒀜𒋾𒅀𒌋\n𒋫𒍑𒋙 𒅁𒋳𒁺\...   
1  𒂗𒅀 𒀴𒅗\n𒂄𒈬 𒀀𒈾 𒈗 𒂗𒐊 𒀭𒀝 𒌋 𒀭\n𒌓𒎌 𒋤𒎌 𒂅𒌒 𒊮𒁉 𒂅𒌒\n𒆪 𒆗𒉡...   
2  𒌋𒐈 𒌑 𒈬𒇻𒌋 𒐉 𒌑 𒂼 𒂍𒆗𒉌 𒂍𒆗𒉌\n𒈬𒇻𒌋 𒐉 𒌑 𒂼 𒂍 𒍇𒇻\n𒈬𒇻𒌋 𒐈 ...   
3  𒈬 𒀀\n𒌋𒅗 𒇻𒎌 𒊭\n𒅖𒁓𒀀\n𒀀𒋫𒀀 𒇻𒎌 𒋬 𒊮\n𒋫𒅗𒆷𒅆 𒈗 𒁁\n𒇻𒎌 𒀀𒈾...   
4  𒉡𒊌 𒁹 𒈨 𒎙 𒄑𒃡𒎌\n𒊭 𒈗 𒀀𒇷𒂊\n𒋗𒌑 𒄿𒋢𒄯 𒅅𒁲𒁉𒀀\n𒈠𒀀 𒄑𒃡𒎌 𒊩𒅆𒌋...   
5  𒀀𒈾 𒈗 𒁁𒉌𒅀\n𒀴𒅗 𒁹𒀸𒋩𒂦𒅆𒅀\n𒇻𒌑 𒂄𒈬 𒀀𒈾 𒈗 𒁁𒉌𒅀\n𒇽𒃲𒐐𒅀 𒇽𒃲𒐐𒅀...   

                                         translation  total_tokens  \
0  [To the king], my [lord: your servant Kiṣir-Aš...           107   
1  [To the king m]y lord: your servant [Hunnî]. G...           179   
2  Height[ 1]3 cubits (6,5 m), width four cubits ...            67   
3  [To the king, my lord: your servant NN. Good h...           119   
4  I (wrote him): "Where are the king's 120 logs?...           105   
5  To the king, my lord: your servant Aššur-dur-p...           165   

   tokens_without_broken  
0                     26  
1                    124  
2                     47  
3                     70  
4                     49  
5                    162

In [8]:
# Export the combined dataset
out = OUTPUT_DIR
out.mkdir(parents=True, exist_ok=True)

path_jsonl = out / "combined_dataset.jsonl"
path_csv = out / "combined_dataset.csv"

combined.to_json(path_jsonl, orient="records", lines=True, force_ascii=False)
combined.to_csv(path_csv, index=False)

print(f"Saved to:")
print(f"  {path_jsonl}  ({path_jsonl.stat().st_size/1024:.1f} KB)")
print(f"  {path_csv}  ({path_csv.stat().st_size/1024:.1f} KB)")
print(f"\n📁 All output files in {out}:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved to:
  C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\output\combined_dataset.jsonl  (41.1 KB)
  C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\output\combined_dataset.csv  (39.9 KB)
📁 All output files in C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\output:
  combined_dataset.csv                      39.9 KB
  combined_dataset.jsonl                    41.1 KB
  saao_saa01.csv                            26.8 KB
  saao_saa01.jsonl                          28.2 KB


## 5. Quick analysis of the data

In [7]:
print("Texts by project:")
print(combined["project"].value_counts().to_string())

print("\nTexts by provenance:")
print(combined["provenance"].value_counts().to_string())

print("\nTexts by period:")
print(combined["period"].value_counts().to_string())

print(f"\nAvg tokens per text: {combined['total_tokens'].mean():.0f}")
print(f"Total tokens: {combined['total_tokens'].sum()}")

Texts by project:
project
cams/gkab    3
Texts by provenance:
provenance
Uruk    3
Texts by period:
period
Neo-Assyrian    6

Avg tokens per text: 124
Total tokens: 742
